# 남은 시나리오 행 추출: 보건교사·경기도 (검수 직전까지)

어제 `02_학교_행동_추출.ipynb`와 **같은 청크·스키마·프롬프트(`school_action_v1`)·자동 검사**로, 아직 추출하지 않은 행 3개를 추출한다. 코드는 `src/preprocess/school/extract.py`로 옮겨 두었다(02 노트북 셀 extract-004·006·008·010·012와 같은 규칙).

| TEMP 청크 | 문서 | 행 | 공식 ID(B04에서 부여) |
|---|---|---|---|
| `TEMP:school_manual:B0025:02` | 교육부 매뉴얼 PDF 36쪽 | 보건(담당)교사 | `moe_school_manual_v3:p0036:t01r02` |
| `TEMP:gyeonggi_guide:B0071:03` | 경기도 가이드북 PDF 54쪽 | 담임교사 | `gyeonggi_school_guide_2025:p0054:t01r01` |
| `TEMP:gyeonggi_guide:B0071:04` | 경기도 가이드북 PDF 54쪽 | 보건(담당)교사 | `gyeonggi_school_guide_2025:p0054:t01r02` |

- **API 비용 주의:** `EXTRACTED_RUN = None`이면 LLM을 호출해 새 실행 폴더를 만든다. 이미 추출한 폴더 이름을 넣으면 호출하지 않고 그 폴더를 읽는다.
- 결과는 Git 제외 `data/interim/school/run_*`에 저장되고, 모든 후보는 `PENDING`이다. 승인은 마지막 셀의 검수 화면에서 사람이 한다.
- 이 노트북은 `notebooks/school` 폴더에서 실행한다.

In [1]:
import os
import sys
from collections import Counter
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

ROOT = Path("../..").resolve()
sys.path.insert(0, str(ROOT / "src"))
load_dotenv(ROOT / ".env")

from index.school.dictionaries import load_dictionaries
from index.school.normalize import load_artifacts, load_raw_texts, precheck, read_jsonl
from preprocess.school.extract import (
    DEFAULT_MODEL,
    annotate_candidates,
    build_scenario_chunks,
    extract_chunk,
    load_concept_draft,
    make_model,
    save_run,
)
from preprocess.school.markdown_data import load_school_data

BASE_DIR = ROOT / "data" / "interim" / "school"
APPROVED_RUN = BASE_DIR / "run_20261104T045311_203944Z"   # 어제 추출·오늘 승인 완료(담임교사 행)
EXTRACTED_RUN = "run_20261104T054859_870962Z"            # 11/4 추출 완료. None이면 새로 LLM 호출
RETRY_RUN = "run_20261104T055051_403958Z"                # 11/4 재추출 완료. None이면 재추출 셀이 LLM 호출

_, sources_df, blocks_df = load_school_data(ROOT)
chunks = build_scenario_chunks(blocks_df, sources_df)
# 어제 실행과 같은 TEMP ID·같은 원문 조각인지 확인한다(다르면 B01에서 두 실행을 합칠 수 없음).
assert chunks == read_jsonl(APPROVED_RUN / "chunks.jsonl"), "어제 청크와 다릅니다"
done = {c["source_chunk_id"] for c in read_jsonl(APPROVED_RUN / "steps_raw.jsonl")}
targets = [c for c in chunks if c["chunk_id"] not in done]
display(pd.DataFrame([{"chunk_id": c["chunk_id"], "PDF": c["pdf_page"], "인쇄": c["printed_page"],
                       "역할 행": c["role_label"], "PDF 대조": c["page_verification"]} for c in targets]))
assert len(targets) == 3

,chunk_id,PDF,인쇄,역할 행,PDF 대조
0,TEMP:school_manual:B0025:02,36,22,보건(담당)교사,VERIFIED_PDF
1,TEMP:gyeonggi_guide:B0071:03,54,44,담임교사,VERIFIED_PDF
2,TEMP:gyeonggi_guide:B0071:04,54,44,보건(담당)교사,VERIFIED_PDF


## LLM 추출

청크마다 한 번 호출하고, 호출 실패·파싱 실패는 한 번 더 시도한다. 한 청크가 실패해도 다음 청크로 넘어가고 실패 사실은 `run.jsonl`에 남는다.

In [2]:
concept_draft = load_concept_draft(ROOT)
chunks_by_id = {c["chunk_id"]: c for c in chunks}
if EXTRACTED_RUN is None:
    model_name = os.getenv("SCHOOL_EXTRACTION_MODEL", DEFAULT_MODEL)   # 어제와 같은 모델
    model = make_model(model_name)
    runs, candidates = [], []
    for chunk in targets:
        record, rows = extract_chunk(chunk, concept_draft, model, model_name)
        runs.append(record)
        candidates += rows
        usage = record["token_usage"] or {}
        print(f"{chunk['chunk_id']} {chunk['role_label']}: {record['status']} / 후보 {len(rows)}개"
              f" / 재시도 {record['retry_count']} / 토큰 {usage.get('total_tokens')}")
    candidates = annotate_candidates(candidates, chunks_by_id, concept_draft)
    run_dir = save_run(BASE_DIR, chunks, candidates, runs)
else:
    run_dir = BASE_DIR / EXTRACTED_RUN
    for record in read_jsonl(run_dir / "run.jsonl"):
        usage = record["token_usage"] or {}
        print(f"{record['source_chunk_id']}: {record['status']} / 재시도 {record['retry_count']}"
              f" / 토큰 {usage.get('total_tokens')} (저장된 기록)")
print("실행 폴더:", run_dir.relative_to(ROOT))

TEMP:school_manual:B0025:02: SUCCESS / 재시도 0 / 토큰 6179 (저장된 기록)
TEMP:gyeonggi_guide:B0071:03: SUCCESS / 재시도 0 / 토큰 6286 (저장된 기록)
TEMP:gyeonggi_guide:B0071:04: SUCCESS / 재시도 0 / 토큰 5474 (저장된 기록)
실행 폴더: data\interim\school\run_20261104T054859_870962Z


## 검수 전 자동 점검 (검수 참고용)

사람 검수를 뺀 채로 B01 형식·B02 원문·B03 사전 검사를 돌린 결과다.
- `auto_reasons`가 있는 후보는 **승인해도 정규화 단계에서 보류된다.** 반려하거나 '행동 수정안'으로 고칠지 판단한다.
- `concept`가 `NEW_CONCEPT`면 승인해도 개념 확정 전까지 적재되지 않는다(개념 정책 hold).
- 통과해도 의미·조건·담당이 원문과 맞는지는 사람이 확인해야 한다.

In [3]:
books = load_dictionaries()
pd.set_option("display.max_colwidth", 90)


def show_precheck(folder):
    artifacts = load_artifacts(folder)
    checks = precheck(artifacts, load_raw_texts(ROOT, artifacts["chunks"]), books)
    display(pd.DataFrame([{**row, "auto_reasons": "; ".join(row["auto_reasons"]),
                           "check_reasons": "; ".join(c["check_reasons"]),
                           "quote_ok": c["quote_check_pass"], "stage": c["stage"],
                           "step": c["step_order"], "modality": c["modality"], "when_text": c["when_text"]}
                          for row, c in zip(checks, artifacts["candidates"], strict=True)]))
    print("청크별 후보:", dict(Counter(c["source_chunk_id"] for c in artifacts["candidates"])))
    print("개념 상태:", dict(Counter(str(row["concept"]) for row in checks)))
    print("자동 점검 걸림:", sum(bool(row["auto_reasons"]) for row in checks), "/", len(checks))
    print("실패한 호출:", [(r["source_chunk_id"], r["status"]) for r in artifacts["failed_runs"]] or "없음")
    return checks


checks = show_precheck(run_dir)

,candidate_index,source_chunk_id,action_text,roles,concept,auto_reasons,check_reasons,quote_ok,stage,step,modality,when_text
0,1,TEMP:school_manual:B0025:02,교사와 진한이가 보건실에 도착한 즉시 손을 씻도록 요청한다.,[HEALTH_TEACHER(EXAMPLE)],NEW_CONCEPT,,예시 단서와 역할 배정의 관계를 PDF에서 확인 필요,True,RESPONSE_1,1,UNSPECIFIED,NaN
1,2,TEMP:school_manual:B0025:02,마스크를 착용한다.,[HEALTH_TEACHER(EXAMPLE)],NEW_CONCEPT,,예시 단서와 역할 배정의 관계를 PDF에서 확인 필요,True,RESPONSE_1,2,UNSPECIFIED,NaN
2,3,TEMP:school_manual:B0025:02,체온을 측정한다.,[HEALTH_TEACHER(EXAMPLE)],NEW_CONCEPT,,예시 단서와 역할 배정의 관계를 PDF에서 확인 필요,True,RESPONSE_1,2,UNSPECIFIED,NaN
3,4,TEMP:school_manual:B0025:02,환자의 증상과 징후를 관찰하고 감염병이 의심되는지를 확인한다.,[HEALTH_TEACHER(EXAMPLE)],DETECT_SYMPTOMATIC,,예시 단서와 역할 배정의 관계를 PDF에서 확인 필요,True,RESPONSE_1,2,UNSPECIFIED,NaN
4,5,TEMP:school_manual:B0025:02,진한이가 의료기관 진료를 받지 않았음을 확인한다.,[HEALTH_TEACHER(EXAMPLE)],NEW_CONCEPT,,예시 단서와 역할 배정의 관계를 PDF에서 확인 필요,True,RESPONSE_1,3,UNSPECIFIED,NaN
5,6,TEMP:school_manual:B0025:02,담임교사에게 보호자에게 연락하도록 요청한다.,"[HEALTH_TEACHER(EXAMPLE), HOMEROOM_TEACHER(EXAMPLE)]",CONTACT_GUARDIAN,표 행 담당(보건(담당)교사)과 다른 역할: 담임교사,예시 단서와 역할 배정의 관계를 PDF에서 확인 필요; 예시 단서와 역할 배정의 관계를 PDF에서 확인 필요,True,RESPONSE_1,3,UNSPECIFIED,진한이가 의료기관 진료를 받지 않았음을 확인하고
6,7,TEMP:school_manual:B0025:02,담임교사에게 의료기관 진료를 받도록 안내할 것을 요청한다.,"[HEALTH_TEACHER(EXAMPLE), HOMEROOM_TEACHER(EXAMPLE)]",REQUEST_MEDICAL_VISIT,표 행 담당(보건(담당)교사)과 다른 역할: 담임교사,예시 단서와 역할 배정의 관계를 PDF에서 확인 필요; 예시 단서와 역할 배정의 관계를 PDF에서 확인 필요,True,RESPONSE_1,3,UNSPECIFIED,진한이가 의료기관 진료를 받지 않았음을 확인하고
7,8,TEMP:school_manual:B0025:02,교무부장에게 연락하여 격리 담당 교사 결정을 요청한다.,"[HEALTH_TEACHER(EXAMPLE), HEAD_OF_ACADEMIC_AFFAIRS(EXAMPLE)]",NEW_CONCEPT,표 행 담당(보건(담당)교사)과 다른 역할: 교무부장,예시 단서와 역할 배정의 관계를 PDF에서 확인 필요; 예시 단서와 역할 배정의 관계를 PDF에서 확인 필요,True,RESPONSE_1,4,UNSPECIFIED,일시적 관찰이 필요하므로
8,9,TEMP:school_manual:B0025:02,진한이를 일시적 관찰실로 이동한다.,[HEALTH_TEACHER(EXAMPLE)],ESCORT_TO_HEALTH_ROOM,,예시 단서와 역할 배정의 관계를 PDF에서 확인 필요,True,RESPONSE_1,5,UNSPECIFIED,NaN
9,10,TEMP:school_manual:B0025:02,부모님이 올 때까지 일시적 관찰실에서 진한이를 일시적으로 보호·관찰한다.,[HEALTH_TEACHER(EXAMPLE)],NEW_CONCEPT,,예시 단서와 역할 배정의 관계를 PDF에서 확인 필요,True,RESPONSE_1,5,UNSPECIFIED,NaN


청크별 후보: {'TEMP:school_manual:B0025:02': 12, 'TEMP:gyeonggi_guide:B0071:03': 10, 'TEMP:gyeonggi_guide:B0071:04': 13}
개념 상태: {'NEW_CONCEPT': 12, 'DETECT_SYMPTOMATIC': 1, 'CONTACT_GUARDIAN': 2, 'REQUEST_MEDICAL_VISIT': 2, 'ESCORT_TO_HEALTH_ROOM': 2, 'CONTACT_HEALTH_TEACHER': 1, 'VENTILATE_DISINFECT': 1, 'HEALTH_EDUCATION': 1, 'None': 13}
자동 점검 걸림: 17 / 35
실패한 호출: 없음


### 결과: 경기도 보건교사 행(`B0071:04`)은 역할 근거 인용 오류

첫 추출에서 `TEMP:gyeonggi_guide:B0071:04`의 후보 13개가 모두 역할 근거(`basis_evidence`)로 **표 제목**("감염병 발생 시나리오 단계 1 주요 조치사항")을 인용했다. 역할은 표 행의 담당 칸에서 인용해야 하므로 B02에서 전부 보류된다. 검수 화면에서는 역할 근거를 고칠 수 없어서 **같은 프롬프트로 이 청크만 다시 추출**한다. 첫 폴더의 이 13개는 검수하지 않고 PENDING으로 두면 정규화에서 자동으로 보류된다.

In [4]:
RETRY_CHUNK_IDS = ["TEMP:gyeonggi_guide:B0071:04"]
if RETRY_RUN is None:
    model_name = os.getenv("SCHOOL_EXTRACTION_MODEL", DEFAULT_MODEL)
    model = make_model(model_name)
    runs, candidates = [], []
    for chunk in (chunks_by_id[cid] for cid in RETRY_CHUNK_IDS):
        record, rows = extract_chunk(chunk, concept_draft, model, model_name)
        runs.append(record)
        candidates += rows
        usage = record["token_usage"] or {}
        print(f"{chunk['chunk_id']} {chunk['role_label']}: {record['status']} / 후보 {len(rows)}개"
              f" / 재시도 {record['retry_count']} / 토큰 {usage.get('total_tokens')}")
    retry_dir = save_run(BASE_DIR, chunks, annotate_candidates(candidates, chunks_by_id, concept_draft), runs)
else:
    retry_dir = BASE_DIR / RETRY_RUN
print("재추출 폴더:", retry_dir.relative_to(ROOT))
retry_checks = show_precheck(retry_dir)

TEMP:gyeonggi_guide:B0071:04 보건(담당)교사: SUCCESS / 후보 12개 / 재시도 0 / 토큰 5898
재추출 폴더: data\interim\school\run_20261104T055051_403958Z


,candidate_index,source_chunk_id,action_text,roles,concept,auto_reasons,check_reasons,quote_ok,stage,step,modality,when_text
0,1,TEMP:gyeonggi_guide:B0071:04,교사와 진한이가 보건실에 도착한 즉시 손을 씻도록 요청한다.,[HEALTH_TEACHER(EXPLICIT)],NEW_CONCEPT,,,True,RESPONSE_1,1,UNSPECIFIED,NaN
1,2,TEMP:gyeonggi_guide:B0071:04,마스크를 착용한다.,[HEALTH_TEACHER(EXPLICIT)],NEW_CONCEPT,,,True,RESPONSE_1,2,UNSPECIFIED,NaN
2,3,TEMP:gyeonggi_guide:B0071:04,체온을 측정한다.,[HEALTH_TEACHER(EXPLICIT)],DETECT_SYMPTOMATIC,,,True,RESPONSE_1,2,UNSPECIFIED,NaN
3,4,TEMP:gyeonggi_guide:B0071:04,환자의 증상과 징후를 관찰하고 감염병이 의심되는지를 확인한다.,[HEALTH_TEACHER(EXPLICIT)],DETECT_SYMPTOMATIC,,,True,RESPONSE_1,2,UNSPECIFIED,NaN
4,5,TEMP:gyeonggi_guide:B0071:04,진한이가 의료기관 진료를 받지 않았음을 확인한다.,[HEALTH_TEACHER(EXPLICIT)],NEW_CONCEPT,,,True,RESPONSE_1,3,UNSPECIFIED,NaN
5,6,TEMP:gyeonggi_guide:B0071:04,보호자에게 연락한다.,[HOMEROOM_TEACHER(EXPLICIT)],CONTACT_GUARDIAN,표 행 담당(보건(담당)교사)과 다른 역할: 담임교사,,True,RESPONSE_1,3,UNSPECIFIED,NaN
6,7,TEMP:gyeonggi_guide:B0071:04,보호자에게 의료기관 진료를 받도록 안내한다.,[HOMEROOM_TEACHER(EXPLICIT)],REQUEST_MEDICAL_VISIT,표 행 담당(보건(담당)교사)과 다른 역할: 담임교사,,True,RESPONSE_1,3,UNSPECIFIED,NaN
7,8,TEMP:gyeonggi_guide:B0071:04,교무부장에게 연락하여 격리 담당 교사의 결정을 요청한다.,[HEALTH_TEACHER(EXPLICIT)],NEW_CONCEPT,,,True,RESPONSE_1,4,UNSPECIFIED,일시적 관찰이 필요하므로
8,9,TEMP:gyeonggi_guide:B0071:04,진한이를 일시적 관찰실로 이동한다.,[HEALTH_TEACHER(EXPLICIT)],ESCORT_TO_HEALTH_ROOM,,,True,RESPONSE_1,5,UNSPECIFIED,NaN
9,10,TEMP:gyeonggi_guide:B0071:04,부모님이 올 때까지 일시적 관찰실에서 진한이를 보호·관찰한다.,[HEALTH_TEACHER(EXPLICIT)],NEW_CONCEPT,,,True,RESPONSE_1,5,UNSPECIFIED,부모님이 올 때까지


청크별 후보: {'TEMP:gyeonggi_guide:B0071:04': 12}
개념 상태: {'NEW_CONCEPT': 7, 'DETECT_SYMPTOMATIC': 2, 'CONTACT_GUARDIAN': 1, 'REQUEST_MEDICAL_VISIT': 1, 'ESCORT_TO_HEALTH_ROOM': 1}
자동 점검 걸림: 3 / 12
실패한 호출: 없음


## 검수 참고: 판정·개념 제안표 (제안일 뿐 - 판정은 검수자가 원문을 보고 한다)

검수 화면에서 고칠 수 있는 것은 **판정·행동 문장·개념**이다. 역할(`roles`)·조건(`when_text`)은 고칠 수 없으므로, 역할이 틀린 후보는 반려하고 사유를 적는다. `(대기)`는 승인 대기 개념이다. 골라 두면 3인 승인 뒤 `concepts.json`으로 옮기기만 해도 적재된다(그 전에는 적재 보류).

**첫 추출 `run_20261104T054859_870962Z`**

| # | 후보 | 자동 점검 | 제안 |
|---|---|---|---|
| 1 | 손을 씻도록 요청 (교육부 보건) | 통과 | 승인 · `HAND_HYGIENE`(대기) 또는 `HEALTH_EDUCATION` |
| 2 | 마스크 착용 | 통과 | 승인 · `WEAR_MASK`(대기) |
| 3 | 체온 측정 | 통과 | 승인 · `DETECT_SYMPTOMATIC` (경기도 재추출 3번과 맞춤) |
| 4 | 증상·징후 관찰, 감염병 의심 확인 | 통과 | 승인 · LLM 후보(`DETECT_SYMPTOMATIC`) |
| 5 | 의료기관 진료를 받지 않았음을 확인 | 통과 | 승인 · `CHECK_MEDICAL_RESULT`로 볼지 판단 |
| 6·7 | 담임교사에게 보호자 연락·진료 안내 요청 | 역할 불일치 | **반려** - 역할에 담임교사가 들어감. 이 행의 수행자는 보건교사(요청)이고, 담임의 보호자 연락은 담임 행(a09·a10)에 이미 있음 |
| 8 | 교무부장에게 격리 담당 교사 결정 요청 | 역할 불일치 | **반려** - 수신자 교무부장이 수행 역할로 들어감 |
| 9 | 일시적 관찰실로 이동 | 통과 | 승인 · LLM 후보(`ESCORT_TO_HEALTH_ROOM`: 보건실·일시적 관찰실 이동) |
| 10 | 관찰실에서 보호·관찰 | 통과 | 승인 · `TEMPORARY_OBSERVATION`(대기) |
| 11 | 담당 교사를 통해 상태 주기 확인 | 역할 미등록 | **반려** - 역할이 '담당 교사'(수행자는 보건교사) |
| 12 | 출입 금지 안내판 부착 | 통과 | 승인 · `TEMPORARY_OBSERVATION`(대기) |
| 13~16·21·22 | 경기도 담임 행(교육부 a01~a04·a09·a10과 같은 행동) | 통과 | 승인 · LLM 후보 |
| 17 | 이동 담당 교사 지정 요청 | 통과 | 승인 · `ASSIGN_RESPONSE_STAFF`(대기) |
| 18 | 마스크 각각 착용 | 통과 | `WEAR_MASK`(대기). **조건 확인**: when_text가 '담임교사가 임신부 등 고위험군일 경우'인데 교육부 a06은 조건 없음 - 원문 괄호 범위를 보고 판단 |
| 19 | 관찰실 보호·관찰 | 통과 | 승인 · `TEMPORARY_OBSERVATION`(대기) |
| 20 | 기타 학생 수업 관리 요청 | 통과 | 승인 · `ARRANGE_CLASS_COVERAGE`(대기) |
| 23~35 | 경기도 보건 행 첫 추출 | 역할 근거 오류 | **검수하지 않음** (재추출 폴더로 대체) |

**재추출 `run_20261104T055051_403958Z` (경기도 보건 행)**

| # | 후보 | 자동 점검 | 제안 |
|---|---|---|---|
| 1 | 손을 씻도록 요청 | 통과 | 승인 · `HAND_HYGIENE`(대기) (첫 추출 1번과 같게) |
| 2 | 마스크 착용 | 통과 | 승인 · `WEAR_MASK`(대기) |
| 3·4 | 체온 측정 / 증상 관찰·의심 확인 | 통과 | 승인 · LLM 후보(`DETECT_SYMPTOMATIC`) |
| 5 | 진료를 받지 않았음을 확인 | 통과 | 첫 추출 5번과 같은 판단 |
| 6·7 | 보호자 연락 / 진료 안내 (역할 담임교사) | 역할 불일치 | **반려** - 첫 추출 6·7번과 같은 이유 |
| 8 | 교무부장에게 격리 담당 교사 결정 요청 | 통과 | 승인 · `ASSIGN_RESPONSE_STAFF`(대기) |
| 9 | 일시적 관찰실로 이동 | 통과 | 승인 · LLM 후보 |
| 10 | 관찰실 보호·관찰 | 통과 | 승인 · `TEMPORARY_OBSERVATION`(대기) |
| 11 | 상태 주기 확인 (역할 '담당 교사') | 역할 미등록 | **반려** |
| 12 | 안내판 부착 | 통과 | 승인 · `TEMPORARY_OBSERVATION`(대기) |

어제 승인한 `run_20261104T045311_203944Z`의 5~8번(개념 미확정)도 이 화면에서 다시 열어 개념만 골라 저장할 수 있다(제안: 5 `ASSIGN_RESPONSE_STAFF`, 6 `WEAR_MASK`, 7 `TEMPORARY_OBSERVATION`, 8 `ARRANGE_CLASS_COVERAGE`).


## 검수 화면

첫 셀과 이 셀만 실행하면 된다(`EXTRACTED_RUN`·`RETRY_RUN`이 채워져 있으면 중간 셀을 실행해도 LLM을 부르지 않는다). 드롭다운에서 두 폴더(첫 추출·재추출)를 차례로 골라 판정을 저장한다. 첫 폴더의 `B0071:04` 후보(23~35번)는 건너뛴다. 승인을 마치면 `school_normalize_bundle.ipynb`의 `RUN_DIRS`에 두 폴더를 넣고 실행한다.

In [5]:
from preprocess.school.review_ui import create_review_ui

display(create_review_ui(BASE_DIR))